# Track 1 error analysis: committed baselines

This notebook reads the pinned English dev reference and committed predictions for cues+logreg, both+logreg, and DeBERTa seeds 13, 42, and 1234. It does not train, tune, or rewrite any earlier experiment. Tested calculations live in `dico_nli.analysis`; this notebook writes only to `experiments/2026-10-02-error-analysis-track1/`.

A longer entailing phrase means the length cue agrees; a longer entailed phrase means it misleads. Equal-length pairs and EQUIVALENCE/NEGATIVE_OTHER rows are uninformative. Cue-slice F1 is descriptive; SoftCons and HardCons are not recomputed on filtered subsets because filtering may break reversible pairs.

In [ ]:
from datetime import date
import json

import matplotlib
matplotlib.use("Agg")
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score

from dico_nli.analysis import PAIR_OUTCOMES, analyze_predictions, read_prediction_file
from dico_nli.data import LABELS, REPO_ROOT, load_reference
from dico_nli.experiments import experiment_dir, write_config
from dico_nli.plotting import figure, paper_theme, save_figure
from dico_nli.scoring import Scores

TRACK = 1
RUN_DATE = date(2026, 10, 2)
OVERWRITE = True
SEED = 13
SEEDS = [13, 42, 1234]
EXACT_COMMAND = "PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/05_error_analysis_track1.ipynb"
TRIVIAL_RUN = REPO_ROOT / "experiments" / "2026-09-29-trivial-baselines-track1"
ML_RUN = REPO_ROOT / "experiments" / "2026-09-29-ml-baseline-track1"
DEBERTA_RUN = REPO_ROOT / "experiments" / "2026-09-29-deberta-base-track1"
PREDICTION_SOURCES = {
    "cues+logreg": ML_RUN / "predictions" / "cues+logreg.csv",
    "both+logreg": ML_RUN / "predictions" / "both+logreg.csv",
    **{f"DeBERTa seed {seed}": DEBERTA_RUN / "predictions" / f"seed{seed}.csv" for seed in SEEDS},
}
ALL_SCORE_SOURCES = {
    "majority": TRIVIAL_RUN / "scores" / "majority" / "scores.json",
    "random seed 13": TRIVIAL_RUN / "scores" / "random" / "scores.json",
    "cues+logreg": ML_RUN / "scores" / "cues+logreg" / "scores.json",
    "tfidf+logreg": ML_RUN / "scores" / "tfidf+logreg" / "scores.json",
    "both+logreg": ML_RUN / "scores" / "both+logreg" / "scores.json",
    "cues+gbdt": ML_RUN / "scores" / "cues+gbdt" / "scores.json",
    **{f"DeBERTa seed {seed}": DEBERTA_RUN / "scores" / f"seed{seed}" / "scores.json" for seed in SEEDS},
}
SYSTEMS = list(PREDICTION_SOURCES)
CUE_STATUSES = ["agrees", "misleads", "uninformative"]
paper_theme()

## 1. Load the dev reference and predictions

The official dev reference is pinned at commit `588968e`; each saved prediction file must cover its 660 rows exactly once.

In [ ]:
dev = load_reference(TRACK, "dev")
assert len(dev) == 660, f"Expected 660 dev rows, found {len(dev)}"

run_dir = experiment_dir("error-analysis-track1", day=RUN_DATE, overwrite=OVERWRITE)
tables_dir, figures_dir = run_dir / "tables", run_dir / "figures"
tables_dir.mkdir(exist_ok=True)
figures_dir.mkdir(exist_ok=True)

predictions = {}
for system, path in PREDICTION_SOURCES.items():
    assert path.is_file(), f"Missing committed prediction file: {path}"
    predictions[system] = read_prediction_file(path)
analyses = {system: analyze_predictions(dev, values) for system, values in predictions.items()}

write_config(
    run_dir, seed=SEED, track=TRACK, command=EXACT_COMMAND,
    extra={
        "analysis": "committed dev predictions; no model fitting",
        "run_date": RUN_DATE.isoformat(),
        "deberta_seeds": SEEDS,
        "cue_model": "cues+logreg",
        "prediction_sources": {name: path.relative_to(REPO_ROOT).as_posix() for name, path in PREDICTION_SOURCES.items()},
        "score_sources": {name: path.relative_to(REPO_ROOT).as_posix() for name, path in ALL_SCORE_SOURCES.items()},
        "length_tokenizer": "dico_nli.eda.tokens; word count",
    },
)
print(f"Loaded {len(dev)} dev rows for {len(analyses)} prediction systems.")

## 2. Overall scores and pair outcomes

Overall metrics are read from each system's saved official-scorer report. Pair outcomes are recomputed from the reference and predictions, once per reversible pair.

In [ ]:
overall_rows = []
for system, report_path in ALL_SCORE_SOURCES.items():
    assert report_path.is_file(), f"Missing saved scorer report: {report_path}"
    report = json.loads(report_path.read_text(encoding="utf-8"))
    scores = Scores.from_report(report_path)
    overall_rows.append({
        "system": system,
        "weighted_f1": scores.weighted_f1,
        "macro_f1": report["classification"]["macro_f1"],
        "accuracy": report["classification"]["accuracy"],
        "soft_cons": scores.soft_cons,
        "hard_cons": scores.hard_cons,
    })

pair_rows = []
for system in SYSTEMS:
    pair_frame, _ = analyses[system]
    counts = pair_frame["outcome"].value_counts()
    for outcome in PAIR_OUTCOMES:
        count = int(counts.get(outcome, 0))
        pair_rows.append({"system": system, "outcome": outcome, "count": count, "share": count / len(pair_frame)})

overall_scores = pd.DataFrame(overall_rows)
pair_outcomes = pd.DataFrame(pair_rows)
overall_scores.to_csv(tables_dir / "overall_scores.csv", index=False)
pair_outcomes.to_csv(tables_dir / "pair_outcomes.csv", index=False)
display(overall_scores.round(3))
display(pair_outcomes.pivot(index="system", columns="outcome", values="count").reindex(SYSTEMS))

## 3. Scores by length-cue status

`agrees` and `misleads` are the direct H2 comparison. `uninformative` preserves ties and non-directional labels. These are classification metrics on subsets, not official consistency metrics.

In [ ]:
cue_rows = []
for system in SYSTEMS:
    _, frame = analyses[system]
    for status in CUE_STATUSES:
        subset = frame.loc[frame["cue_status"] == status]
        cue_rows.append({
            "system": system,
            "cue_status": status,
            "n": len(subset),
            "weighted_f1": f1_score(subset["gold_label"], subset["predicted_label"], labels=list(LABELS), average="weighted", zero_division=0),
            "macro_f1": f1_score(subset["gold_label"], subset["predicted_label"], labels=list(LABELS), average="macro", zero_division=0),
            "accuracy": accuracy_score(subset["gold_label"], subset["predicted_label"]),
        })
cue_slice_scores = pd.DataFrame(cue_rows)
cue_slice_scores.to_csv(tables_dir / "cue_slice_scores.csv", index=False)
display(cue_slice_scores.pivot(index="system", columns="cue_status", values="weighted_f1").reindex(SYSTEMS).round(3))
display(cue_slice_scores.pivot(index="system", columns="cue_status", values="n").reindex(SYSTEMS))

## 4. The three-seed DeBERTa hard core

The hard core is the instance-ID intersection of errors made by all three fixed seeds. The table also records whether the cue-only logistic baseline misses each instance.

In [ ]:
deberta_maps = [predictions[f"DeBERTa seed {seed}"] for seed in SEEDS]
hard_core_ids = {
    instance.instance_id for instance in dev
    if all(model[instance.instance_id] != instance.label for model in deberta_maps)
}
cue_predictions = predictions["cues+logreg"]
hard_core_rows = []
for instance in dev:
    if instance.instance_id not in hard_core_ids:
        continue
    hard_core_rows.append({
        "instance_id": instance.instance_id,
        "pair_id": instance.pair_id,
        "text1": instance.text1,
        "text2": instance.text2,
        "gold_label": instance.label,
        **{f"deberta_seed_{seed}": predictions[f"DeBERTa seed {seed}"][instance.instance_id] for seed in SEEDS},
        "cues_logreg_prediction": cue_predictions[instance.instance_id],
        "cues_logreg_wrong": cue_predictions[instance.instance_id] != instance.label,
    })
hard_core = pd.DataFrame(hard_core_rows)
hard_core_summary = pd.DataFrame([{
    "all_seed_wrong_instances": len(hard_core),
    "cue_model_also_wrong": int(hard_core["cues_logreg_wrong"].sum()),
    "cue_model_also_wrong_share": float(hard_core["cues_logreg_wrong"].mean()),
}])
hard_core.to_csv(tables_dir / "deberta_all_seeds_wrong.csv", index=False)
hard_core_summary.to_csv(tables_dir / "deberta_all_seeds_wrong_summary.csv", index=False)
display(hard_core_summary)
display(hard_core.head(10))

## 5. Twenty inspected seed-13 errors

The taxonomy is guided by the pilot paper's manual error analysis, summarized in [`docs/research/apaolaza-2026-logical-coherence.md`](../../docs/research/apaolaza-2026-logical-coherence.md): prepositional phrases, near-synonymy, and chunking/phrase fragments. Numeral/quantifier scope and one-word NEGATIVE_OTHER contrasts are added because they recur in this dev sample. The 20 items below are assigned one primary category by manual inspection.

These are selected errors, not a random sample, so category counts describe this table only and are not estimates of population prevalence.

In [ ]:
ERROR_ANNOTATIONS = [
    ("dico_phrasis_0000141__en-en__original", "prepositional_relation", "Meaning depends on the preposition or its attachment."),
    ("dico_phrasis_0000370__en-en__flipped", "prepositional_relation", "The pair changes the preposition around a shared event phrase."),
    ("dico_phrasis_0000994__en-en__flipped", "prepositional_relation", "Temporal meaning uses a preposition and changed word order."),
    ("dico_phrasis_0002487__en-en__original", "prepositional_relation", "A locative preposition changes an otherwise overlapping negative pair."),
    ("dico_phrasis_0000350__en-en__flipped", "numeral_or_quantifier", "Resolve an all-people quantifier against a numeric range."),
    ("dico_phrasis_0000619__en-en__flipped", "numeral_or_quantifier", "Compare a group description with an exact cardinality."),
    ("dico_phrasis_0001386__en-en__original", "numeral_or_quantifier", "The numeral changes the scope from a plural group to a singular category."),
    ("dico_phrasis_0002244__en-en__original", "numeral_or_quantifier", "The same number is written as a word and as a digit."),
    ("dico_phrasis_0000190__en-en__flipped", "near_synonym_or_taxonomy", "Color and number variants are close but not identical descriptions."),
    ("dico_phrasis_0000198__en-en__flipped", "near_synonym_or_taxonomy", "Resolve a specific food item against a broader near-synonym."),
    ("dico_phrasis_0001551__en-en__flipped", "near_synonym_or_taxonomy", "The relation between a demographic noun phrase and its group label is subtle."),
    ("dico_phrasis_0002900__en-en__original", "near_synonym_or_taxonomy", "Occupational labels overlap but are not freely interchangeable."),
    ("dico_phrasis_0002436__en-en__original", "one_word_negative_contrast", "The child/daughter substitution has NEGATIVE_OTHER as its gold label."),
    ("dico_phrasis_0002560__en-en__original", "one_word_negative_contrast", "A single adjective substitution separates two child descriptions."),
    ("dico_phrasis_0003063__en-en__original", "one_word_negative_contrast", "A single adjective substitution separates crowded from public."),
    ("dico_phrasis_0002712__en-en__original", "one_word_negative_contrast", "A changed modifier identifies a different organization in a negative pair."),
    ("dico_phrasis_0000024__en-en__flipped", "chunking_or_fragment", "A short inflected verb fragment must align with its base form."),
    ("dico_phrasis_0000375__en-en__flipped", "chunking_or_fragment", "Short verb phrases package the event with different complements."),
    ("dico_phrasis_0000421__en-en__original", "chunking_or_fragment", "The nominal phrase gains a possessive and a head-modifying request fragment."),
    ("dico_phrasis_0000616__en-en__flipped", "chunking_or_fragment", "A compound noun phrase is aligned with a differently chunked screen description."),
]
category_order = ["prepositional_relation", "numeral_or_quantifier", "near_synonym_or_taxonomy", "one_word_negative_contrast", "chunking_or_fragment"]
category_rationales = {
    "prepositional_relation": "Preposition choice or attachment changes the relation.",
    "numeral_or_quantifier": "Cardinality and quantifier scope need explicit reasoning.",
    "near_synonym_or_taxonomy": "Close lexical relations are not always equivalence or entailment.",
    "one_word_negative_contrast": "A single lexical substitution preserves overlap but changes the gold relation.",
    "chunking_or_fragment": "Short phrase fragments and different chunk boundaries obscure the relation.",
}
annotations = pd.DataFrame(ERROR_ANNOTATIONS, columns=["instance_id", "category", "rationale"])
seed13_errors = analyses["DeBERTa seed 13"][1].loc[lambda frame: ~frame["is_correct"]]
error_examples = seed13_errors.merge(annotations, on="instance_id", validate="one_to_one")
if len(error_examples) != 20 or annotations["instance_id"].nunique() != 20:
    raise ValueError(f"Expected 20 distinct annotated seed-13 errors, found {len(error_examples)}")
error_examples["category"] = pd.Categorical(error_examples["category"], categories=category_order, ordered=True)
error_examples = error_examples.sort_values(["category", "instance_id"])
category_counts = error_examples.groupby("category", observed=True).size().reindex(category_order, fill_value=0).rename("count").reset_index()
category_counts["rationale"] = category_counts["category"].map(category_rationales)
error_examples.to_csv(tables_dir / "seed13_error_examples.csv", index=False)
category_counts.to_csv(tables_dir / "seed13_error_category_counts.csv", index=False)
display(error_examples[["category", "instance_id", "text1", "text2", "gold_label", "predicted_label"]])
display(category_counts)

## 6. Figures

Figures use the existing project plotting helpers and are saved as vector PDFs with PNG previews.

In [ ]:
outcome_pivot = pair_outcomes.pivot(index="system", columns="outcome", values="share").reindex(index=SYSTEMS, columns=PAIR_OUTCOMES).fillna(0)
fig, ax = figure(columns=2, height=3.5)
outcome_pivot.plot(kind="barh", stacked=True, ax=ax, color=["#1baf7a", "#2a78d6", "#eb6834", "#4a3aa7"])
ax.set(title="Dev reversible-pair outcomes", xlabel="Share of pairs", ylabel="", xlim=(0, 1))
ax.legend(title="Outcome", loc="lower center", bbox_to_anchor=(0.5, -0.45), ncol=2)
save_figure(fig, figures_dir / "pair_outcomes")

cue_pivot = cue_slice_scores.pivot(index="system", columns="cue_status", values="weighted_f1").reindex(index=SYSTEMS, columns=CUE_STATUSES)
fig, ax = figure(columns=2, height=3.5)
cue_pivot.plot(kind="bar", ax=ax, color=["#2a78d6", "#eb6834", "#1baf7a"])
ax.set(title="Weighted F1 by length-cue status", xlabel="", ylabel="Weighted F1", ylim=(0, 1))
ax.tick_params(axis="x", labelrotation=20)
ax.legend(title="Cue status")
save_figure(fig, figures_dir / "cue_slice_weighted_f1")

fig, ax = figure(columns=1, height=3.1)
counts_for_plot = category_counts.groupby("category", observed=True)["count"].sum().reindex(category_order)
ax.barh(counts_for_plot.index.astype(str), counts_for_plot.values, color="#2a78d6")
ax.set(title="Manual categories in 20 selected seed-13 errors", xlabel="Examples", ylabel="")
save_figure(fig, figures_dir / "seed13_error_categories")
print(f"Wrote figures under {figures_dir}")

## 7. Findings and run record

The report is generated from the computed tables. It reports H2 gaps directly and treats the hand-selected error categories as qualitative examples.

In [ ]:
deberta_slice_mean = cue_slice_scores.loc[cue_slice_scores["system"].str.startswith("DeBERTa")].groupby("cue_status")["weighted_f1"].mean()
cue_only_slice = cue_slice_scores.loc[cue_slice_scores["system"] == "cues+logreg"].set_index("cue_status")
h2_gaps = (deberta_slice_mean - cue_only_slice["weighted_f1"]).reindex(CUE_STATUSES)
cue_findings = "\n".join(
    f"| {status} | {int(cue_only_slice.loc[status, 'n'])} | {cue_only_slice.loc[status, 'weighted_f1']:.3f} | {deberta_slice_mean.loc[status]:.3f} | {h2_gaps.loc[status]:+.3f} |"
    for status in CUE_STATUSES
)
pair_findings = "\n".join(
    f"- **{system}:** " + ", ".join(
        f"{outcome} {int(row['count'])}/{int(pair_outcomes.loc[pair_outcomes['system'] == system, 'count'].sum())}"
        for outcome, row in pair_outcomes.loc[pair_outcomes["system"] == system].set_index("outcome").reindex(PAIR_OUTCOMES).iterrows()
    )
    for system in SYSTEMS
)
category_findings = "\n".join(f"| {row.category} | {int(row.count)} | {row.rationale} |" for row in category_counts.itertuples(index=False))
findings = f"""# Assignment 1 error analysis: track 1 dev

## Data and method

This analysis uses the 660-row English dev reference at data commit `588968e610197ddc4c440314701cbc587afc4c1b` and committed predictions for cues+logreg, both+logreg, and DeBERTa seeds 13, 42, and 1234. No model was retrained or selected on dev. Full-set metrics use saved official-scorer reports; subset metrics are classification-only because cue filtering may break reversible pairs.

## Pair outcomes

{pair_findings}

`both_directions_correct` means both gold labels were predicted. `consistent_but_wrong` means predictions obey reversal but are not both gold-correct. The other categories are inconsistent with one or neither direction correct.

## H2: scores by length-cue status

| Cue status | n | Cues + logreg weighted F1 | DeBERTa mean weighted F1 | Encoder minus cue |
|---|---:|---:|---:|---:|
{cue_findings}

The largest DeBERTa-minus-cue weighted-F1 gap is in **{h2_gaps.idxmax()}** ({h2_gaps.max():+.3f}). Equal-length and non-directional examples are uninformative to the length cue and remain separate. The misleading-cue subset contains only {int(cue_only_slice.loc['misleads', 'n'])} rows, so its gap is descriptive rather than a stable estimate.

## Shared DeBERTa errors

All three DeBERTa seeds misclassify **{len(hard_core)}** of 660 dev instances. Cues+logreg also misclassifies **{int(hard_core['cues_logreg_wrong'].sum())}** ({hard_core['cues_logreg_wrong'].mean():.1%}). Full list: `tables/deberta_all_seeds_wrong.csv`.

## Twenty inspected seed-13 errors

| Primary category | Count | Rationale |
|---|---:|---|
{category_findings}

The category families are guided by the pilot paper's manual analysis (Apaolaza et al., 2026; see [`docs/research/apaolaza-2026-logical-coherence.md`](../../docs/research/apaolaza-2026-logical-coherence.md)); numeral/quantifier scope and one-word negative contrasts are included because they recur in this sample. These errors were manually selected, not sampled randomly; counts describe these 20 only. Phrases and gold/predicted labels: `tables/seed13_error_examples.csv`.

## Full-set scores

See `tables/overall_scores.csv` for weighted F1, macro F1, accuracy, SoftCons, and HardCons for every existing system. See `tables/cue_slice_scores.csv` for n and classification metrics on each cue group.

## Caveats

SoftCons and HardCons are not computed on cue-filtered subsets because a subset may remove one direction of a reversible pair. Manual categories involve judgment and are not corpus-wide category rates. Predictions are fixed; uncertainty beyond the three DeBERTa seeds is not estimated.
"""
(run_dir / "findings.md").write_text(findings, encoding="utf-8")
print(findings)
print(f"Run outputs: {run_dir}")